# 演習9. 安全な終了

## シーン

ここまでの演習では、毎回こう書いてきました。

```cpp
for (int i = 0; i < N; i++) { ... }      // N個で終わり、と決め打ち
```

**現実にはそうはいきません。**

- 動画が何フレームあるかは、読み終わるまで分かりません
- カメラ入力なら、そもそも終わりがありません
- ユーザが `q` を押したら、その場でやめたい

つまり **「もう来ない」を、どうやって下流に伝えるか**という問題です。
そして、伝えそこねたときに何が起きるかを、まず見ます。

## 9-1. 【予測クイズ】「もう来ない」を伝えないと

読む側は10フレームで終わります。使う側は、何枚来るか知らないので
「来るかぎり処理する」と書きます。**これがいちばん自然な書き方です。**

```cpp
while (true) {
    int f = q.pop();      // 空なら待つ
    ...処理する...
}
```

**実行する前に予測してください。** このプログラムはどうなるでしょうか。
異常終了しますか、正常に終わりますか、それとも別のことが起きますか。

（止まったままにならないよう、5秒で強制終了させます。）

In [ ]:
%%writefile ex09a.cpp
#include <iostream>
#include <thread>
#include <vector>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
using namespace std::chrono;

// ---- 演習5で組み立てたキュー（ここでは中身は読まなくてよい） ----
template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t capacity) : capacity_(capacity) {}
    void push(const T& v) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_push_.wait(lk, [this] { return q_.size() < capacity_; });
        q_.push(v); lk.unlock(); can_pop_.notify_one();
    }
    T pop() {
        std::unique_lock<std::mutex> lk(mtx_);
        can_pop_.wait(lk, [this] { return !q_.empty(); });
        T v = q_.front(); q_.pop(); lk.unlock(); can_push_.notify_one();
        return v;
    }
private:
    std::queue<T> q_;
    std::size_t capacity_;
    mutable std::mutex mtx_;
    std::condition_variable can_pop_, can_push_;
};

void wait_ms(int ms) { std::this_thread::sleep_for(milliseconds(ms)); }

int main() {
    BoundedQueue<int> q(4);

    // 読む側：10フレームで動画が終わる（何枚あるかは、読み終わるまで分からない）
    std::thread reader([&] {
        for (int i = 0; i < 10; i++) { wait_ms(50); q.push(i); }
        std::cout << "読む側 : 動画が終わったので、読むのをやめる\n" << std::flush;
    });

    // 使う側：何枚来るか知らないので、来るかぎり処理し続ける
    std::thread worker([&] {
        while (true) {
            int f = q.pop();                       // ← 11枚目を永久に待つことになる
            std::cout << "  処理した : " << f << "\n" << std::flush;
        }
    });

    reader.join();
    std::cout << "読む側の join() は返った\n" << std::flush;
    worker.join();                                 // ← ここから先へ進めない
    std::cout << "ここには到達しない\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex09a.cpp -o ex09a
!timeout 5 ./ex09a; echo "終了コード=$? （124 なら止まった）"

### 結果 ―― 10枚とも正しく処理して、それから固まる

**仕事は全部きちんと終わっています。** 10枚とも処理されました。
それでもプログラムは終わりません。

使う側は11枚目を待って `pop()` の中で眠っています。誰も起こしません。
そのため `worker.join()` が永久に返らず、`main` も終われません。

演習3・4で見たものと、まったく同じ壊れ方です。

- エラーは出ない
- 異常終了もしない
- **結果はすべて正しい**
- ただ、終わらない

> **「処理は終わったのに、プログラムが終わらない」は、終了設計の欠落。**

### 伝え方は2通りある

「もう来ない」の伝え方には、大きく2つのやり方があります。

- **キューに「閉じた」という状態を持たせる**（9-2）
- **「これで終わり」という目印のデータを流す**（9-3、**番兵**と呼びます）

どちらも実務で使われます。順に見ていきます。

## 9-2. キューを「閉じる」

演習5のキューに、`close()` を足します。足すのは3つだけです。

```cpp
bool closed_ = false;                       // ① 状態を1つ持つ

bool pop(T& out) {
    std::unique_lock<std::mutex> lk(mtx_);
    can_pop_.wait(lk, [this] { return !q_.empty() || closed_; });   // ② 条件を2つに
    if (q_.empty()) return false;           // 閉じていて、もう残っていない
    ...
    return true;
}

void close() {                              // ③ 閉じる操作
    { std::lock_guard<std::mutex> g(mtx_); closed_ = true; }
    can_pop_.notify_all();                  // 待っている全員を起こす
    can_push_.notify_all();
}
```

演習4の発展課題3でやったことと、まったく同じ形です。
**述語を「データが来た、または、もう来ない」の2条件にする**のがすべてです。

使う側はこうなります。

```cpp
int f;
while (q1.pop(f)) {        // false が返ったら「もう来ない」
    ...処理する...
    q2.push(f);
}
q2.close();                // ← 自分の下流にも伝える
```

**最後の1行が肝心です。** 終了は**段から段へ、順に伝わっていきます。**

**実行する前に予測してください。**

- 10枚とも表示されるでしょうか。それとも途中で打ち切られるでしょうか
- 3つのスレッドは、どういう順で終わるでしょうか

In [ ]:
%%writefile ex09b.cpp
#include <iostream>
#include <thread>
#include <vector>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
using namespace std::chrono;

// ---- 演習5のキューに「閉じる」を足したもの ----
template <typename T>
class ClosableQueue {
public:
    explicit ClosableQueue(std::size_t capacity) : capacity_(capacity) {}

    // 入れる。閉じられていたら何もせず false
    bool push(const T& v) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_push_.wait(lk, [this] { return q_.size() < capacity_ || closed_; });
        if (closed_) return false;
        q_.push(v);
        lk.unlock(); can_pop_.notify_one();
        return true;
    }

    // 取り出す。取れたら true。閉じられていて、かつ空なら false
    bool pop(T& out) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_pop_.wait(lk, [this] { return !q_.empty() || closed_; });
        if (q_.empty()) return false;              // 閉じていて、もう残っていない
        out = q_.front(); q_.pop();
        lk.unlock(); can_push_.notify_one();
        return true;
    }

    // もう入れない、と宣言する
    void close() {
        { std::lock_guard<std::mutex> g(mtx_); closed_ = true; }
        can_pop_.notify_all();                     // 待っている全員を起こす
        can_push_.notify_all();
    }

private:
    std::queue<T> q_;
    std::size_t capacity_;
    bool closed_ = false;
    mutable std::mutex mtx_;
    std::condition_variable can_pop_, can_push_;
};

void wait_ms(int ms) { std::this_thread::sleep_for(milliseconds(ms)); }

int main() {
    ClosableQueue<int> q1(4), q2(4);

    std::thread reader([&] {
        for (int i = 0; i < 10; i++) { wait_ms(50); q1.push(i); }
        std::cout << "Read  : 動画が終わった -> q1 を閉じる\n" << std::flush;
        q1.close();                                // ← ここが要
    });

    std::thread inferer([&] {
        int f;
        while (q1.pop(f)) {                        // false が返ったら「もう来ない」
            wait_ms(60);
            q2.push(f);
        }
        std::cout << "Infer : q1 が閉じた -> q2 を閉じる\n" << std::flush;
        q2.close();                                // ← 終了を下流へ伝える
    });

    std::thread shower([&] {
        int f;
        while (q2.pop(f)) { wait_ms(30); std::cout << "  表示した : " << f << "\n" << std::flush; }
        std::cout << "Show  : q2 が閉じた -> 終わる\n" << std::flush;
    });

    reader.join(); inferer.join(); shower.join();
    std::cout << "\nすべてのスレッドが join() できた\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex09b.cpp -o ex09b && ./ex09b

### 結果 ―― 10枚とも処理してから、上流から順に終わる

```
Read  : 動画が終わった -> q1 を閉じる
  ... （まだ 6,7,8 が処理されている）
Infer : q1 が閉じた -> q2 を閉じる
  表示した : 9
Show  : q2 が閉じた -> 終わる
すべてのスレッドが join() できた
```

**1枚も捨てていません。** ここが大事なところです。

`pop` の中身をもう一度見てください。

```cpp
can_pop_.wait(lk, [this] { return !q_.empty() || closed_; });
if (q_.empty()) return false;
```

閉じられていても、**キューに残っていれば先に取り出します。**
`false` を返すのは「閉じていて、かつ空」のときだけです。

もし条件の順序を逆にして

```cpp
if (closed_) return false;      // ← これは間違い
```

と書くと、**閉じた瞬間に残りを全部捨てます。**

> **「閉じた」と「空になった」は別のこと。**
> **終わるのは、閉じて、かつ空になったとき。**

### 終了は下流へ伝わる

```
Read が読み終わる
   └─▶ q1.close()
          └─▶ Infer の while が抜ける
                 └─▶ q2.close()
                        └─▶ Show の while が抜ける
```

段が10個あっても同じです。**各段が「自分の下流を閉じる」だけ**で、
終了は端まで伝わります。

そして重要なのは、**この形なら `join()` が必ず返る**ということです。
`join()` が返らない原因は、ほぼ例外なく「終了が伝わっていない」ことです。

## 9-3. 番兵（sentinel）を流す

もう1つのやり方は、**「これで終わり」という目印のデータを、普通のデータと同じように流す**
というものです。この目印を **番兵（sentinel）** あるいは **poison pill** と呼びます。

キットには手を入れません。**演習5のキューのまま**使えます。

```cpp
const int END = -1;                    // これが番兵

// 読む側
for (...) q1.push(i);
q1.push(END);                          // 最後に目印を流す

// 使う側
while (true) {
    int f = q1.pop();
    if (f == END) break;               // 目印が来たら終わり
    ...処理する...
    q2.push(f);
}
q2.push(END);                          // 下流へ渡す
```

**ただし、受け取る人が複数いると話が変わります。**
番兵は1個しかないので、受け取れるのは1人だけです。
残りの人は永久に待つことになります。

そこで **人数分の番兵を流します。**

**実行する前に予測してください。** Infer が2人のとき、
Show 側は番兵を何個受け取ることになるでしょうか。

In [ ]:
%%writefile ex09c.cpp
#include <iostream>
#include <thread>
#include <vector>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
using namespace std::chrono;

// ---- 演習5で組み立てたキュー（ここでは中身は読まなくてよい） ----
template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t capacity) : capacity_(capacity) {}
    void push(const T& v) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_push_.wait(lk, [this] { return q_.size() < capacity_; });
        q_.push(v); lk.unlock(); can_pop_.notify_one();
    }
    T pop() {
        std::unique_lock<std::mutex> lk(mtx_);
        can_pop_.wait(lk, [this] { return !q_.empty(); });
        T v = q_.front(); q_.pop(); lk.unlock(); can_push_.notify_one();
        return v;
    }
private:
    std::queue<T> q_;
    std::size_t capacity_;
    mutable std::mutex mtx_;
    std::condition_variable can_pop_, can_push_;
};

const int NWORKER = 2;              // Infer は2人
const int END = -1;                 // これが番兵（終わりの目印）
void wait_ms(int ms) { std::this_thread::sleep_for(milliseconds(ms)); }

int main() {
    BoundedQueue<int> q1(4), q2(4);     // 演習5のキューのまま。close() は無い

    std::thread reader([&] {
        for (int i = 0; i < 10; i++) { wait_ms(50); q1.push(i); }
        std::cout << "Read  : 動画が終わった -> 番兵を " << NWORKER << " 個流す\n" << std::flush;
        for (int k = 0; k < NWORKER; k++) q1.push(END);   // ← 人数分いる
    });

    std::vector<std::thread> inferers;
    for (int k = 0; k < NWORKER; k++)
        inferers.emplace_back([&, k] {
            while (true) {
                int f = q1.pop();
                if (f == END) break;                     // 番兵を受け取ったら抜ける
                wait_ms(60);
                q2.push(f);
            }
            std::cout << "Infer" << k << ": 番兵を受け取った -> 番兵を1個流して終わる\n" << std::flush;
            q2.push(END);                                // 受け取った1個を下流へ渡す
        });

    std::thread shower([&] {
        int seen = 0;
        while (seen < NWORKER) {                         // 番兵を人数分そろえる
            int f = q2.pop();
            if (f == END) { seen++; continue; }
            wait_ms(30);
            std::cout << "  表示した : " << f << "\n" << std::flush;
        }
        std::cout << "Show  : 番兵が " << NWORKER << " 個そろった -> 終わる\n" << std::flush;
    });

    reader.join();
    for (auto& t : inferers) t.join();
    shower.join();
    std::cout << "\nすべてのスレッドが join() できた\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex09c.cpp -o ex09c && ./ex09c

### 結果 ―― 番兵は人数分いる

Infer が2人なので、Read は番兵を2個流します。
各 Infer は自分が受け取った1個を下流へ渡すので、Show には**2個**届きます。
Show は2個そろうまで待ってから終わります。

**この「人数分」の管理が、番兵方式のいちばん厄介なところ**です。
段ごとに人数が違えば、流す個数も変わります。

### 番兵方式と close() 方式の比べ方

**番兵の利点**

- **キューに手を入れなくてよい。** ライブラリのキューをそのまま使えます
- **順序が保たれる。** 番兵はデータと同じ列に並ぶので、
  「番兵より前のデータは必ず処理される」ことが自動的に保証されます
- **データと同じ道を通る**ので、考えることが減ります

**番兵の弱点**

- **人数分そろえる管理**が要ります。人数を変えたら、流す数も直さなければなりません
- **途中でやめられません。** 番兵は列の最後尾に並ぶので、
  前のデータが全部さばけるまで届きません。
  「いますぐ止めたい」には向きません
- **データ型に「ありえない値」が必要**です。`int` なら `-1` でよくても、
  画像そのものを流している場合は、それ用の印を作る必要があります

**close() の利点・弱点は、ちょうど裏返し**です。

- 誰でも、いつでも、**列に並ばずに**閉じられます（途中でやめるのに向く）
- そのかわり、キュー自身に手を入れる必要があります

> **通常の終了なら、どちらでもよい。**
> **「途中でやめる」が要るなら、`close()` 方式。**

本番のプログラムがどちらを使っているかは、**キューのクラスに `close` に相当するものが
あるかどうか**を見れば分かります。

## まとめ ―― 終了のチェックリスト

パイプラインを書いたら、次の5つを確かめてください。

**① すべてのスレッドに「抜ける道」があるか**

`while (true)` の中に `break` か `return` の条件がありますか。
`pop()` が永久に待つ形になっていませんか。

**② 終了が下流まで伝わるか**

各段が、自分の下流に終了を伝えていますか。
段が4つあれば、伝達も3回起きるはずです。

**③ 残っているデータを捨てていないか**

「閉じた」と「空になった」を混同していませんか。
終わってよいのは、**閉じて、かつ空になった**ときだけです。

**④ 待っている全員を起こしているか**

`close()` の中は `notify_all()` ですか。`notify_one()` では1人しか終われません
（演習4の発展課題3）。

**⑤ すべてのスレッドを `join()` しているか**

起動したスレッドは、必ずどこかで `join()` します（演習1の発展課題5）。

> **`join()` が返らないなら、①〜④のどれかが抜けている。**

デバッグのときは、各段の終了時に1行表示してみてください。
**どこまで伝わって、どこで止まったか**がすぐ分かります。

## 発展課題

1. 9-3 で、番兵を**1個しか流さなかった**らどうなるでしょうか。Infer は2人です。
   どのスレッドが、どういう理由で止まるか説明してください。

2. 9-2 の `close()` 方式で、**Infer を2人**にしたとします。
   `q2.close()` を呼ぶのは誰でしょうか。
   **2人とも自分が終わったときに呼んだら**、何が起きるでしょうか。

3. 「ユーザが `q` を押したので、いますぐやめたい」を実現したいとします。
   - `close()` はどのスレッドから呼べばよいでしょうか
   - **キューに残っているデータ**は、処理してから終わるべきでしょうか、捨てるべきでしょうか
   - 「上流のキューだけ閉じる」のと「全部のキューを閉じる」のでは、何が違うでしょうか

4. `close()` したあとに `push()` を呼ぶと `false` が返ります。
   **この戻り値を無視する**コードを書くと、どんな問題が起きるでしょうか。

5. あるスレッドの中で**例外が飛んだ**らどうなるでしょうか。
   そのスレッドは終わりますが、`join()` はどうなりますか。下流はどうなりますか。

6. `detach()` を使えば `join()` は要らなくなります。
   それでもこの問題が解決しない理由を説明してください。